# 08 — Volatility Regime Backtest

## Why this project exists

A trader may ask: **"Does a simple trend strategy behave differently when VIX is high versus low?"**

This notebook combines a moving average trend signal with a volatility regime filter. The aim is not to build a production strategy, but to show how I test a market hypothesis in a structured way.

In [ ]:
!pip install yfinance plotly -q

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import plotly.express as px

pd.options.display.float_format = "{:,.4f}".format

## 1. Download SPY and VIX data

SPY gives equity exposure. VIX provides a simple volatility regime proxy.

In [ ]:
data = yf.download(["SPY", "^VIX"], start="2005-01-01", auto_adjust=True, progress=False)["Close"].dropna()
spy = data["SPY"]
vix = data["^VIX"]

df = pd.DataFrame({"spy": spy, "vix": vix}).dropna()
df.tail()

## 2. Create trend and volatility regime features

Signal design:

- Trend signal: SPY above 200-day moving average
- Volatility regime: VIX below or above 20

To avoid look-ahead bias, signals are shifted by one day.

In [ ]:
df["return"] = df["spy"].pct_change()
df["ma_200"] = df["spy"].rolling(200).mean()
df["trend_raw"] = (df["spy"] > df["ma_200"]).astype(int)
df["low_vix_raw"] = (df["vix"] < 20).astype(int)

df["trend_signal"] = df["trend_raw"].shift(1)
df["trend_low_vix_signal"] = (df["trend_raw"] * df["low_vix_raw"]).shift(1)

df["buy_hold_return"] = df["return"]
df["trend_return"] = df["trend_signal"] * df["return"]
df["trend_low_vix_return"] = df["trend_low_vix_signal"] * df["return"]
df = df.dropna()
df.tail()

## 3. Evaluate performance

I compare three cases:

1. Buy and hold
2. Trend filter only
3. Trend filter + low VIX filter

The question is whether the VIX filter improves risk-adjusted returns or simply removes too much exposure.

In [ ]:
def max_drawdown(equity):
    return (equity / equity.cummax() - 1).min()

def summarize(ret):
    equity = (1 + ret).cumprod()
    ann_return = ret.mean() * 252
    ann_vol = ret.std() * np.sqrt(252)
    sharpe = ann_return / ann_vol if ann_vol != 0 else np.nan
    exposure = (ret != 0).mean()
    return pd.Series({
        "annual_return": ann_return,
        "annual_volatility": ann_vol,
        "sharpe_ratio": sharpe,
        "max_drawdown": max_drawdown(equity),
        "market_exposure": exposure,
    })

summary = pd.DataFrame({
    "buy_and_hold": summarize(df["buy_hold_return"]),
    "trend_only": summarize(df["trend_return"]),
    "trend_low_vix": summarize(df["trend_low_vix_return"]),
}).T
summary

In [ ]:
equity = pd.DataFrame({
    "buy_and_hold": (1 + df["buy_hold_return"]).cumprod(),
    "trend_only": (1 + df["trend_return"]).cumprod(),
    "trend_low_vix": (1 + df["trend_low_vix_return"]).cumprod(),
})
px.line(equity, title="Equity curves by strategy variant").show()
px.line(df[["vix"]], title="VIX volatility regime proxy").show()

## 4. My conclusion

This project demonstrates how to test whether a market regime filter adds value.

What I would discuss with a trader:

- Does the filter reduce drawdowns?
- Does it miss too many upside days?
- Is VIX < 20 too simplistic?
- Would a rolling VIX percentile be better than a fixed threshold?

## Next iterations

- Test VIX percentiles instead of fixed thresholds
- Add transaction costs
- Add parameter grid
- Test across QQQ, IWM and sector ETFs
- Add walk-forward validation